In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,END, START
from typing import TypedDict

In [10]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"

In [11]:
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [12]:
class ConvType(TypedDict):
    topic:str
    outline:str
    blog:str

In [13]:
def outline_generation(state: ConvType):
    topic = state["topic"]

    template = PromptTemplate(
        input_variables=["topic"],
        template="You are a helpful AI that gives short but engaging GenZ-style answers. Use slang to give a comprehensive and engaging outline for this topic. Topic: {topic}"
    )

    prompt = template.format(topic=topic)
    response=model.invoke(prompt).content

    return {"outline": response}

In [14]:
def blog_generation(state: ConvType):
    topic = state["topic"]
    outline=state['outline']

    template = PromptTemplate(
        input_variables=["topic","outline"],
        template="You are a helpful AI that gives short but engaging GenZ-style answers. Use slang to give a comprehensive blog based on the {topic} and the outline {outline}"
    )

    prompt = template.format(topic=topic, outline=outline)
    response=model.invoke(prompt).content

    return {"blog": response}

In [15]:
graph= StateGraph(ConvType)

graph.add_node('outline_generation',outline_generation)
graph.add_node('blog_generation',blog_generation)
graph.add_edge(START,"outline_generation")
graph.add_edge("outline_generation","blog_generation")
graph.add_edge("blog_generation",END)

In [16]:
workflow=graph.compile()
final_node=workflow.invoke({'topic':"blackholes"})

In [17]:
print(final_node["outline"])

Hey! So basically, black holes are these super duper strong gravitational fields in space where not even light can escape. They're like cosmic vacuum cleaners that suck everything in their path. There's the regular black hole, then there's the supermassive one at the center of galaxies, which is like a galactic monster eating up stars and gas.

Black holes are formed when massive stars die and collapse in on themselves. It's like if you squished a beach ball until it became a tiny dot - that's kind of what happens to a star. The gravity gets so intense that nothing can get out, not even light, which is why we call them "black."

Scientists study black holes using telescopes and other instruments because they're invisible. They can't be seen directly, but their effects on nearby stars and gas clouds give away their presence.

So, next time you look up at the night sky, remember that there might be these cosmic mysteries lurking out there, just waiting to be explored!


In [18]:
print(final_node["blog"])

Hey y'all! So basically, black holes are these super duper strong gravitational fields in space where not even light can escape. They're like cosmic vacuum cleaners that suck everything in their path. There's the regular black hole, then there's the supermassive one at the center of galaxies, which is like a galactic monster eating up stars and gas.

Black holes are formed when massive stars die and collapse in on themselves. It's like if you squished a beach ball until it became a tiny dot - that's kind of what happens to a star. The gravity gets so intense that nothing can get out, not even light, which is why we call them "black."

Scientists study black holes using telescopes and other instruments because they're invisible. They can't be seen directly, but their effects on nearby stars and gas clouds give away their presence.

So, next time you look up at the night sky, remember that there might be these cosmic mysteries lurking out there, just waiting to be explored! 🌌✨
